# 基于tensor的C++的 SIMD 编程
从本节开始，我们将系统学习 Tensor API 的编程范式，深入理解、掌握布局、内存、张量三层抽象的构建与使用方式，并全面了解 Copy与Mmad两类原子操作的参数化与调用机制。这些理论知识将为后续开发高性能算子奠定坚实基础。

本节学习大纲如下：
- 环境准备
- 什么是Tensor API
- Layout的介绍及使用
- Tensor的介绍：张量的创建与访问
- 数据移动（Copy）：CopyAtom 的构建、参数化与执行、支持的搬运路径
- 矩阵计算（Mmad）：MmadAtom 的构建、参数化与执行
- 完整样例开发：矩阵乘算子

---


## 1. 环境准备

正式开始学习之前，先要对Jupyter环境进行初始化。以下代码完成了初始化并将环境中的变量导入Jupyter环境，同时完成了代码目录的创建。保证能正常导入代码以及使用bisheng编译器，完成算子的开发及编译。

In [ ]:
import os
!mkdir -p Sources
import subprocess
import os

result = subprocess.run(
    ['bash', '-l', '-c', 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'],
    capture_output=True, text=True
)
for line in result.stdout.strip().split('\n'):
    line = line.strip()
    if "=" in line and not line.startswith(("#", " ")):
        key, value = line.split("=", 1)
        os.environ[key] = value
print("\n🎉 Environment initialization process completed successfully!")

## 2. 什么是Tensor API
在Ascend C矩阵计算开发中，开发者经常需要关注数据位于哪里、如何排布、不同排布之间如何搬运。如果直接使用C API编写代码，容易出现几个不便之处：多维坐标需要手动换算为线性偏移，切片和子块访问需要反复计算地址，搬运和计算接口的参数组合较多，不同数据位置、布局格式和版本对应的实现路径也不容易统一管理。这会让代码更依赖经验，既影响可读性，也增加了调试和复用成本。

为了解决这些问题，提供了基于Ascend C的C++模板抽象库的张量编程接口。使用Layout描述数据的逻辑形状、步幅和布局格式，用Tensor把数据地址、存储位置和Layout组合起来，使开发者可以按照坐标、切片和张量对象来表达数据访问，而不是手动维护底层地址计算；同时将数据搬运和矩阵乘加等操作抽象为统一接口，并在编译期结合数据位置、布局模式、数据类型和架构版本完成实现路径选择。减少了重复的索引计算，提升代码的可读性、可维护性和复用性。

Tensor API当前仅支持Ascend 950PR/Ascend 950DT。

<div style="text-align: left;">
  <img src="./images/tensor_api_arch.png" alt="Tensor API接口分类示意图" width="700">
  <p>图 1 Tensor API接口分类示意图</p>
</div>
Tensor API的接口主要可以分为Tensor、Algorithm、Atom和Arch四类。分别对应算子编程中的四类问题：数据如何表达、操作如何调用、操作参数如何组织，以及底层实现如何匹配。

- **Tensor类接口**："数据是什么、怎么访问"。它以Tensor为核心，通过组合Engine和Layout统一描述底层地址、存储位置、逻辑形状、步幅和布局格式，并提供坐标访问、切片、子Tensor构造以及Shape/Stride获取等能力。借助这类接口，开发者可以用张量视图表达数据块，避免在代码中散落大量手写偏移计算。

- **Algorithm类接口**："操作怎么调用"。它提供Copy()、Mmad()等通用算法入口，把常见的数据搬运和矩阵计算封装成简洁的函数调用。开发者在编写算子主体时，可以优先使用这类统一入口组织数据流，而不用直接展开每一种底层实现的调用细节。

- **Atom类接口**："这个操作带哪些特征参数"。它将搬运和计算操作封装为CopyAtom、MmadAtom等原子对象，组织操作类型、默认参数、可配置参数和参数展开逻辑，并通过with等机制支持运行时参数配置。这样既保留了对搬运模式、矩阵计算模式等细节的控制能力，又避免把复杂参数直接暴露到每一次算法调用中。

- **Arch类接口**："最终匹配哪条硬件实现路径"。它面向具体硬件架构封装底层指令，并根据数据位置、Layout分形、数据类型和版本号等条件进行静态分发，选择对应的搬运或计算实现。对上层开发者而言，这意味着同一类Copy或Mmad表达可以在满足约束的前提下自动落到合适的硬件路径上，从而把注意力更多放在Tensor构造、数据布局和计算流程本身。


## 3. Layout介绍和使用
### 3.1 Layout的层次化表述
我们使用Shape和Stride来表达Layout排布格式，Shape用于表达Layout形状，Stride则用于区分不同的排布。比如下图中的行优先和列优先排布：

- 行优先:Shape \(2, 4\)  Stride \(4, 1\)
- 列优先:Shape \(2, 4\)  Stride \(1, 2\)

图中每个方格中的数字表示该位置元素在内存中按顺序排列时的下标。对于相同的矩阵位置，排布方式不同时，其在内存中的顺序可能是不同的，例如，矩阵坐标 \(1, 0\) 在行优先和列优先的情况下，对应元素在内存中的顺序分别是4和1。
<div style="text-align: left;">
  <img src="./images/行优先排布.png" alt="行优先排布" width="500">
  <p>图 2 行优先排布</p>
</div>

<div style="text-align: left;">
  <img src="./images/列优先排布.png" alt="列优先排布" width="500">
  <p>图 3 列优先排布</p>
</div>

通常Shape或者Stride中的元素是一个单独的正整数（如上文的示例），但当数据存在多层嵌套排布时，单一整数难以表达，可以采用层次化表述法。层次化表述法的思路是：将数据排布拆解为多个层次，每个层次独立描述一层矩阵的布局，通过层次嵌套来表达复杂的内存排布。在Tensor API的四维Layout中，这一思想体现为将Shape和Stride的每个维度元素从一个整数扩展为一个二元组，每个元组中第一个值描述内层矩阵，第二个值描述外层矩阵。

例如在下图中，Layout的Shape和Stride分别为：

- Shape\(\(2, 3\), \(2, 4\)\)
- Stride\(\(1, 4\), \(2, 12\)\)

<div style="text-align: left;">
  <img src="./images/层次化表述法.png" alt="层次化表述法" width="500">
  <p>图 4 层次化表述法</p>
</div>


图中展示了两层矩阵：内层矩阵为内部用灰色线包裹的矩阵，外层矩阵为将内层矩阵视为一个元素时黑色线包裹的矩阵。

Shape的第一个元素描述行方向的形状，\(2, 3\)表示内层矩阵和外层矩阵的行数分别为2和3；Shape的第二个元素描述的是列方向上的形状，\(2, 4\)表示内层矩阵和外层矩阵的列数分别为2和4。

Stride中的每个元素与Shape中的元素对应，表示该对应维度下，相邻元素首地址在内存地址上的间隔，图片中用箭头表示了每个维度相邻元素的首地址间隔。

### 3.2 常见的Layout分形

在基于Ascend C进行矩阵编程场景下，会用到以下几种常用的Layout，这些格式都采用上文介绍的层次化表述法来表达。要求有内外层两层矩阵，其中行排布可以看作Z字形排布，列排布可以看作N字形排布，Layout的分形名称从前到后依次描述从外层到内层的排布顺序，比如NZLayout格式表示外层为N字形排布，内层为Z字形排布。四维Layout的具体表达方式如下：

```cpp
Layout = ((Shape): (Stride))
Shape = ((ShapeRow0, ShapeRow1), (ShapeColumn0, ShapeColumn1))
Stride = ((StrideRow0, StrideRow1), (StrideColumn0, StrideColumn1))
```

常见的分形对应位置的值如下表所示，其中C0_ELEMENT = Std::Int<32/sizeof(T)>{}。

<table>
<thead>
<tr><th style="text-align: left;">LayoutFormatPattern</th><th style="text-align: left;">类型</th><th style="text-align: left;">ShapeRow0</th><th style="text-align: left;">ShapeRow1</th><th style="text-align: left;">ShapeColumn0</th><th style="text-align: left;">ShapeColumn1</th><th style="text-align: left;">StrideRow0</th><th style="text-align: left;">StrideRow1</th><th style="text-align: left;">StrideColumn0</th><th style="text-align: left;">StrideColumn1</th></tr>
</thead>
<tbody>
<tr><td style="text-align: left;">NZLayoutPtn</td><td style="text-align: left;">T</td><td style="text-align: left;">Int&lt;16&gt;{}</td><td style="text-align: left;">ceil_div(row, Std::Int&lt;16&gt;{})</td><td style="text-align: left;">C0_ELEMENT</td><td style="text-align: left;">ceil_div(column, C0_ELEMENT)</td><td style="text-align: left;">C0_ELEMENT</td><td style="text-align: left;">C0_ELEMENT * Std::Int&lt;16&gt;{}</td><td style="text-align: left;">Int&lt;1&gt;{}</td><td style="text-align: left;">C0_ELEMENT * ceil_align(row, Std::Int&lt;16&gt;{})</td></tr>
<tr><td style="text-align: left;">ZNLayoutPtn</td><td style="text-align: left;">T</td><td style="text-align: left;">C0_ELEMENT</td><td style="text-align: left;">ceil_div(row, C0_ELEMENT)</td><td style="text-align: left;">Std::Int&lt;16&gt;{}</td><td style="text-align: left;">ceil_div(column, Std::Int&lt;16&gt;{})</td><td style="text-align: left;">Std::Int&lt;1&gt;{}</td><td style="text-align: left;">C0_ELEMENT * ceil_align(column, Std::Int&lt;16&gt;{})</td><td style="text-align: left;">C0_ELEMENT</td><td style="text-align: left;">C0_ELEMENT * Std::Int&lt;16&gt;{}</td></tr>
<tr><td style="text-align: left;">DNExtLayoutPtn</td><td style="text-align: left;">T</td><td style="text-align: left;">Std::Int&lt;1&gt;{}</td><td style="text-align: left;">row</td><td style="text-align: left;">Std::Int&lt;1&gt;{}</td><td style="text-align: left;">column</td><td style="text-align: left;">Std::Int&lt;0&gt;{}</td><td style="text-align: left;">Std::Int&lt;1&gt;{}</td><td style="text-align: left;">Std::Int&lt;0&gt;{}</td><td style="text-align: left;">row</td></tr>
<tr><td style="text-align: left;">NDExtLayoutPtn</td><td style="text-align: left;">T</td><td style="text-align: left;">Std::Int&lt;1&gt;{}</td><td style="text-align: left;">row</td><td style="text-align: left;">Std::Int&lt;1&gt;{}</td><td style="text-align: left;">column</td><td style="text-align: left;">Std::Int&lt;0&gt;{}</td><td style="text-align: left;">column</td><td style="text-align: left;">Std::Int&lt;0&gt;{}</td><td style="text-align: left;">Std::Int&lt;1&gt;{}</td></tr>
</tbody>
</table>

以NZLayoutPtn为例，NZ Layout格式的内层分形Shape中，ShapeRow0固定为16行，ShapeColumn0固定为32Byte / sizeof(T)，内层分形的Stride中，StrideRow0固定为32Byte / sizeof(T)，StrideColumn0也固定为1，即内层分形按Z字形组织，外层分形按N字形组织。ShapeRow1需满足row方向16对齐，ShapeColumn1需满足column方向上32B对齐。

<div style="text-align: left;">
    <img src="./images/NZ格式.svg" alt="层次化表述法" width="500">
    <p>图 5 NZ Layout </p>
</div>


下面是一个连续的NZ Layout示例，其中C0_ELEMENT = Std::Int<32>{} / sizeof(T)。

```cpp
Layout = ((Shape) : (Stride))
Shape = ((Std::Int<16>{}, ceil_div(row, Std::Int<16>{})), (C0_ELEMENT, ceil_div(column, C0_ELEMENT)))
Stride = ((C0_ELEMENT, C0_ELEMENT * Std::Int<16>{}), (Std::Int<1>{}, C0_ELEMENT * ceil_align(row, Std::Int<16>{})))
```

### 3.3 Layout的构造

MakeFrameLayout是根据分形简化的Layout构造方式。它将布局模式（LayoutPattern）抽象为模板参数，通过内部根据分形类型自动选择合适的Trait或者用户传入的自定义Trait，生成对应的Shape和Stride。

MakeFrameLayout方法定义如下：
```cpp
// 默认：TraitType = Std::ignore_t
template <typename LayoutPattern, typename TraitType = Std::ignore_t, typename... Args>
__aicore__ inline constexpr decltype(auto) MakeFrameLayout(const Args&... args);
```

Trait是对Layout分形特征的参数化描述，包含两个组成部分：
- type：元素数据类型，决定Layout按什么数据类型解释每个元素（如 half、float、int8_t）。对于 NDExtLayoutPtn / DNExtLayoutPtn 分形，type 为 Std::ignore_t，表示数据排布与元素类型无关。
- C0_ELEMENT：分形内层 column 方向的元素个数，即 ShapeColumn0 的大小，是生成分形 Shape 和 Stride 的关键参数。

当 TraitType 默认为 Std::ignore_t 时，MakeFrameLayout 查找当前 LayoutPattern 对应的默认 Trait，不同分形对应不同的Trait：

<table>
<thead><tr><th>LayoutPattern</th><th>默认 Trait</th><th>type</th><th>C0_ELEMENT</th></tr></thead>
<tbody>
<tr><td>NZLayoutPtn / ZNLayoutPtn / ZZLayoutPtn</td><td>LayoutTrait&lt;uint16_t, Std::Int&lt;16&gt;&gt;</td><td>uint16_t</td><td>Std::Int&lt;16&gt;{}</td></tr>
<tr><td>NDExtLayoutPtn / DNExtLayoutPtn</td><td>LayoutTrait&lt;Std::ignore_t, Std::Int&lt;1&gt;&gt;</td><td>/</td><td>Std::Int&lt;1&gt;{}</td></tr>
</tbody>
</table>

以下三种方式通过MakeFrameLayout构造特性分形的Layout:

```cpp
// 方式1：传入自定义 Trait
struct MyCustomTrait {
    using type = float;
    static constexpr auto C0_ELEMET = Std::Int<16>{};
};

auto layout1 = MakeFrameLayout<NZLayoutPtn, MyCustomTrait>(rows, cols);

// 方式2：通过便捷重载仅覆盖 C0（适用于只需改 C0、不关心 type 的场景）
auto layout2 = MakeFrameLayout<NZLayoutPtn, Std::Int<16>>(rows, cols);  // C0 = 16

// 方式3：不传 TraitType，使用查表默认值
auto layout3 = MakeFrameLayout<NZLayoutPtn>(rows, cols);
auto layout4 = MakeFrameLayout<NDExtLayoutPtn>(rows, cols);
```

理解了Layout的构造方式之后，我们把它放到一个更具体的场景中：假设要实现一个矩阵乘算子，输入矩阵来自GM，计算时需要把数据搬运到片上存储，并按照特定的分形Layout组织后交给矩阵乘单元处理。如果只围绕地址写代码，开发者需要自己维护矩阵块的起始地址、行列偏移、分形排布和搬运路径。

Tensor API提供的解决思路是：先用Layout描述每个矩阵块的逻辑形状和物理排布，再用Tensor把地址和Layout绑定起来，后续通过Copy和Mmad表达数据搬运与矩阵乘加。接下来我们就沿着这个思路逐步构建一个完整的矩阵乘算子，首先创建源文件，引入必要的头文件并定义核函数模板：


In [ ]:
%%writefile Sources/matmul_tensor_api.asc

#include <cstdint>
#include <iostream>
#include <vector>
#include <random>
#include <cmath>
#include <algorithm>
#include "acl/acl.h"
#include "tensor_api/tensor.h"
#include "basic_api/kernel_operator_block_sync_intf.h"

template <uint32_t M, uint32_t K, uint32_t N, uint32_t singleCoreM, uint32_t baseM, uint32_t baseK, uint32_t baseN>
__global__ __cube__ void matmul_custom(__gm__ half* a, __gm__ half* b, __gm__ half* c)
{
    using namespace AscendC::Te;

    // 多核切分：将输出矩阵按M和N方向切分到各Cube核上
    constexpr uint32_t mIter = M / singleCoreM;
    uint32_t mIterIdx = AscendC::GetBlockIdx() % mIter;
    uint32_t nIterIdx = AscendC::GetBlockIdx() / mIter;

    // 创建GM张量，并进行多核切片
    auto gmA = MakeTensor(MakeMemPtr(a), MakeFrameLayout<NDExtLayoutPtn>(M, K));
    auto gmB = MakeTensor(MakeMemPtr(b), MakeFrameLayout<NDExtLayoutPtn>(K, N));
    auto gmC = MakeTensor(MakeMemPtr(c), MakeFrameLayout<NDExtLayoutPtn>(M, N));

    auto globalA = gmA.Slice(MakeCoord(mIterIdx * singleCoreM, 0), MakeShape(baseM, K));
    auto globalB = gmB.Slice(MakeCoord(0, nIterIdx * baseN), MakeShape(K, baseN));
    auto globalC = gmC.Slice(MakeCoord(mIterIdx * singleCoreM, nIterIdx * baseN), MakeShape(baseM, baseN));

    // 定义各级缓存的Layout
    auto l1ALayout = MakeFrameLayout<NZLayoutPtn>(baseM, K);
    auto l1BLayout = MakeFrameLayout<NZLayoutPtn>(K, baseN);
    auto l0ALayout = MakeFrameLayout<NZLayoutPtn>(baseM, baseK);
    auto l0BLayout = MakeFrameLayout<ZNLayoutPtn>(baseK, baseN);
    auto l0CLayout = MakeFrameLayout<NZLayoutPtn>(baseM, baseN);

**代码说明：**
- \_\_global_\_ \_\_cube_\_：标识这是一个在Cube核上执行的设备核函数，矩阵乘使用Cube核以获得最佳性能
- MakeFrameLayout<NDExtLayoutPtn>()：GM侧数据使用四维ND排布NDExtLayoutPtn，即四维行优先排布
- MakeFrameLayout<NZLayoutPtn>()：L1和L0侧使用NZ分形格式，满足Cube单元的数据搬运要求
- MakeFrameLayout<ZNLayoutPtn>()：L0B使用ZN格式（NZ的转置），满足Cube单元的数据搬运要求
- Slice：通过Slice接口实现多核数据切分，每个核通过GetBlockIdx()计算自己负责的数据块

## 4. Tensor介绍

Tensor是Tensor API的核心数据结构，由指针和数据布局组成。一个Tensor对象同时描述"数据放在哪里"和"数据长什么样"。

### 4.1 内存指针介绍
内存指针封装带有内存位置标签的硬件指针，所有指针操作均通过MakeMemPtr函数创建。

- 物理位置

    由Location命名空间定义了 6 种硬件内存位置：

<table>
<thead>
<tr><th style="text-align: left;">位置</th><th style="text-align: left;">语义</th><th style="text-align: left;">地址限定符</th></tr>
</thead>
<tbody>
<tr><td style="text-align: left;">Location::GM</td><td style="text-align: left;">Global Memory</td><td style="text-align: left;">__gm__</td></tr>
<tr><td style="text-align: left;">Location::UB</td><td style="text-align: left;">Unified Buffer</td><td style="text-align: left;">__ubuf__</td></tr>
<tr><td style="text-align: left;">Location::L1</td><td style="text-align: left;">L1 Buffer</td><td style="text-align: left;">__cbuf__</td></tr>
<tr><td style="text-align: left;">Location::L0A</td><td style="text-align: left;">L0A Buffer</td><td style="text-align: left;">__ca__</td></tr>
<tr><td style="text-align: left;">Location::L0B</td><td style="text-align: left;">L0B Buffer</td><td style="text-align: left;">__cb__</td></tr>
<tr><td style="text-align: left;">Location::L0C</td><td style="text-align: left;">L0C Buffer</td><td style="text-align: left;">__cc__</td></tr>
</tbody>
</table>

- 创建指针

    MakeMemPtr是创建硬件指针的统一入口，构造方法如下：

```cpp
// 传入硬件修饰符指针
template <typename Iterator>
__aicore__ inline constexpr auto MakeMemPtr(Iterator iterator)
```


### 4.2 Tensor构造和使用

MakeTensor方法定义如下：

```cpp
template <typename Iterator, typename... Args>
__aicore__ inline constexpr auto MakeTensor(const Iterator& iter, const Args&... args);
```

<div style="text-align: left;">
    <img src="./images/Tensor访问数据和切片示意图.png" alt="Tensor访问数据和切片示意图" width="500">
    <p>图 6 Tensor访问数据和切片示意图 </p>
</div>

- 元素访问

    Tensor通过operator[]支持多维坐标访问，内部由Layout自动完成坐标到线性偏移的换算：

    ```cpp
    auto tensor = MakeTensor(ptr, MakeLayout(MakeShape(16, 16)));
    // 坐标索引
    auto coord = MakeCoord(2, 3);
    auto& val  = tensor[coord];        // ptr[2 * 16 + 3]，Layout 自动计算偏移
    ```

- operator() 切片

    operator()用于创建子张量：从当前Tensor的指定坐标开始，取指定大小的子区域，返回仍保留相同Stride的子Tensor：

    ```cpp
    // 从 (2, 0) 开始，取 1 行 8 列的子矩阵
    auto sub = tensor(MakeCoord(2, 0), MakeShape(1, 8));
    // sub.Shape() = Shape{1, 8}
    // sub.Data()  = tensor.Data() + tensor.Layout()(MakeCoord(2, 0))
    // sub 仍然是完整 Tensor，可继续参与后续操作
    ```

- Slice切片

    Slice提供基于坐标和切分信息的有界切片，支持Shape和Layout两种切分方式：

    ```cpp
    // 从 (4, 4) 开始，切出 3x3 窗口
    auto sliced = Slice(tensor, MakeCoord(4, 4), MakeShape(3, 3));
    // sliced.Shape() = Shape{3, 3}
    ```

- 设置L2CacheHint 

    对于存储在GM位置的Tensor，可通过SetL2CacheHint设置L2缓存策略：

    ```cpp
    tensor.SetL2CacheHint(CacheMode::CACHE_MODE_NORMAL);
    ```


接下来，在源文件中继续添加L1/L0缓冲区的声明和张量构建：

In [ ]:
%%writefile -a Sources/matmul_tensor_api.asc

    // 声明各级缓冲区的内存空间
    __cbuf__ half l1ABuf[baseM * K];
    __cbuf__ half l1BBuf[K * baseN];
    __ca__ half l0ABuf[baseM * baseK];
    __cb__ half l0BBuf[baseK * baseN];
    __cc__ float l0CBuf[baseM * baseN];

    // 创建各级Tensor对象，将内存与Layout绑定
    auto l1ATensor = MakeTensor(MakeMemPtr(l1ABuf), l1ALayout);
    auto l1BTensor = MakeTensor(MakeMemPtr(l1BBuf), l1BLayout);
    auto l0ATensor = MakeTensor(MakeMemPtr(l0ABuf), l0ALayout);
    auto l0BTensor = MakeTensor(MakeMemPtr(l0BBuf), l0BLayout);
    auto l0CTensor = MakeTensor(MakeMemPtr(l0CBuf), l0CLayout);

**代码说明：**
- \_\_cbuf\__：表示被修饰的变量位于L1 Buffer地址空间，MakeMemPtr自动推导为Location::L1
- \_\_ca\__：表示被修饰的变量位于L0A Buffer地址空间，推导为Location::L0A
- \_\_cb\__：表示被修饰的变量位于L0B Buffer地址空间，推导为Location::L0B
- \_\_cc\__：表示被修饰的变量位于L0C Buffer地址空间，推导为Location::L0C
- L0C Buffer上的数据类型为float，而输入L0A Buffer/L0B Buffer上的数据类型为half

至此，所有数据层已准备就绪。下一节将为这些张量配置搬运和计算 Atom。

## 5. Atom介绍

Atom 层位于 Algorithm 层和 Arch 层之间，是 Tensor API 四层架构中的操作封装层。它接收来自 Algorithm 层的统一调用（Copy/Mmad），将Operation（操作类别）与Trait（特征参数）组合为Atom原子对象，最终通过 Arch 层分发到具体的硬件实现路径。

### CopyAtom 操作类型

CopyAtom覆盖了Ascend C中从GM到L0C的全部数据搬运路径，按执行单元分为Vector（AIV）和Cube（AIC）两类：

<table>
<thead>
<tr><th style="text-align: left;">操作类型</th><th style="text-align: left;">执行单元</th><th style="text-align: left;">数据流向</th></tr>
</thead>
<tbody>
<tr><td style="text-align: left;">CopyGM2UB</td><td style="text-align: left;">Vector</td><td style="text-align: left;">GM → UB</td></tr>
<tr><td style="text-align: left;">CopyUB2GM</td><td style="text-align: left;">Vector</td><td style="text-align: left;">UB → GM</td></tr>
<tr><td style="text-align: left;">CopyUB2L1</td><td style="text-align: left;">Vector</td><td style="text-align: left;">UB → L1</td></tr>
<tr><td style="text-align: left;">CopyGM2L1</td><td style="text-align: left;">Cube</td><td style="text-align: left;">GM → L1</td></tr>
<tr><td style="text-align: left;">CopyL12UB</td><td style="text-align: left;">Cube</td><td style="text-align: left;">L1 → UB</td></tr>
<tr><td style="text-align: left;">CopyL12L0A</td><td style="text-align: left;">Cube</td><td style="text-align: left;">L1 → L0A</td></tr>
<tr><td style="text-align: left;">CopyL12L0B</td><td style="text-align: left;">Cube</td><td style="text-align: left;">L1 → L0B</td></tr>
<tr><td style="text-align: left;">CopyL12FB</td><td style="text-align: left;">Cube</td><td style="text-align: left;">L1 → FixBuf</td></tr>
<tr><td style="text-align: left;">CopyL12BT</td><td style="text-align: left;">Cube</td><td style="text-align: left;">L1 → BiasTable</td></tr>
<tr><td style="text-align: left;">CopyL12L0ScaleA</td><td style="text-align: left;">Cube</td><td style="text-align: left;">L1 → L0A(Scale)</td></tr>
<tr><td style="text-align: left;">CopyL12L0ScaleB</td><td style="text-align: left;">Cube</td><td style="text-align: left;">L1 → L0B(Scale)</td></tr>
<tr><td style="text-align: left;">CopyL0C2UB</td><td style="text-align: left;">Cube</td><td style="text-align: left;">L0C → UB</td></tr>
<tr><td style="text-align: left;">CopyL0C2GM</td><td style="text-align: left;">Cube</td><td style="text-align: left;">L0C → GM</td></tr>
</tbody>
</table>

### MmadAtom 操作类型

MmadAtom封装矩阵计算类操作，当前提供一个统一的 MmadOperation，表示AIC Core 上 Cube单元的矩阵乘加运算。

<table>
<thead>
<tr><th style="text-align: left;">操作类型</th><th style="text-align: left;">执行单元</th><th style="text-align: left;">描述</th></tr>
</thead>
<tbody>
<tr><td style="text-align: left;">MmadOperation</td><td style="text-align: left;">Cube</td><td style="text-align: left;">矩阵乘加：C = A × B (+ Bias)</td></tr>
</tbody>
</table>


### 5.1 CopyAtom的构造与使用

#### CopyAtom的构造

Atom的构造方式如下：

```cpp
// MakeCopy + Copy（推荐）
auto atom1 = MakeCopy(CopyGM2L1{}, CopyGM2L1TraitDefault{});
Copy(atom1, dst, src);
```

每个CopyAtom不仅封装了搬运操作类别，还封装了该操作的特征参数 Trait（搬运时用什么配置）。以 CopyL12L0A（L1→L0A 搬运）为例，其Trait决定了是否转置搬运等行为。

通常情况下，用户通常不需要关心 Trait 内部细节，直接用预定义的 TraitDefault即可。

Copy 操作及其默认Trait：

<table>
<thead>
<tr><th style="text-align: left;">执行单元</th><th style="text-align: left;">Copy 操作</th><th style="text-align: left;">默认 Trait</th><th style="text-align: left;">数据流向</th></tr>
</thead>
<tbody>
<tr><td style="text-align: left;">Vector</td><td style="text-align: left;">CopyGM2UB</td><td style="text-align: left;">CopyGM2UBTraitDefault</td><td style="text-align: left;">GM → UB</td></tr>
<tr><td style="text-align: left;">Vector</td><td style="text-align: left;">CopyUB2GM</td><td style="text-align: left;">CopyUB2GMTraitDefault</td><td style="text-align: left;">UB → GM</td></tr>
<tr><td style="text-align: left;">Vector</td><td style="text-align: left;">CopyUB2L1</td><td style="text-align: left;">CopyUB2L1TraitDefault</td><td style="text-align: left;">UB → L1</td></tr>
<tr><td style="text-align: left;">Cube</td><td style="text-align: left;">CopyGM2L1</td><td style="text-align: left;">CopyGM2L1TraitDefault</td><td style="text-align: left;">GM → L1</td></tr>
<tr><td style="text-align: left;">Cube</td><td style="text-align: left;">CopyL12UB</td><td style="text-align: left;">CopyL12UBTraitDefault</td><td style="text-align: left;">L1 → UB</td></tr>
<tr><td style="text-align: left;">Cube</td><td style="text-align: left;">CopyL12L0A</td><td style="text-align: left;">CopyL12L0ATraitDefault</td><td style="text-align: left;">L1 → L0A</td></tr>
<tr><td style="text-align: left;">Cube</td><td style="text-align: left;">CopyL12L0B</td><td style="text-align: left;">CopyL12L0BTraitDefault</td><td style="text-align: left;">L1 → L0B</td></tr>
<tr><td style="text-align: left;">Cube</td><td style="text-align: left;">CopyL0C2GM</td><td style="text-align: left;">CopyL0C2GMTraitDefault</td><td style="text-align: left;">L0C → GM</td></tr>
<tr><td style="text-align: left;">Cube</td><td style="text-align: left;">CopyL0C2UB</td><td style="text-align: left;">CopyL0C2UBTraitDefault</td><td style="text-align: left;">L0C → UB</td></tr>
</tbody>
</table>

#### 定制化构造CopyAtom

当默认Trait无法表达当前搬运需求时，可以定制自己的CopyAtom。定制CopyAtom的核心思路是三步：先选择搬运操作类型（Operation），再选择或定义该操作对应的Trait，最后通过MakeCopy将Operation和Trait组合成Atom。Operation决定"从哪里搬到哪里"，Trait决定"用什么搬运配置"。

以L1到L0A搬运为例，如果希望为某一路搬运单独配置特征参数，可以先定义一个面向该搬运路径的Trait，再构造Atom：

```cpp
// Step 1: 选择搬运路径，CopyL12L0A 表示 L1 → L0A
// Step 2: 定义或选择该路径对应的 Trait，字段以具体API支持的配置项为准
struct MyCopyL12L0ATrait {
    // 在这里描述转置、搬运模式等编译期配置
};

// Step 3: 组合 Operation 和 Trait，得到自定义 CopyAtom
auto myCopyL12L0AAtom = MakeCopy(CopyL12L0A{}, MyCopyL12L0ATrait{});

// Step 4: 后续调用 Copy 时直接复用该 Atom
Copy(myCopyL12L0AAtom, l0aTensor, l1Tensor);
```

如果只是使用该搬运路径的标准行为，则直接使用默认Trait即可：

```cpp
auto copyL12L0AAtom = MakeCopy(CopyL12L0A{}, CopyL12L0ATraitDefault{});
```

实际开发中，可以把常用搬运路径分别封装成不同Atom，例如GM→L1、L1→L0A、L1→L0B、L0C→GM各自使用一个Atom。这样主计算流程只需要关注Copy的源Tensor和目的Tensor，搬运路径及其特征配置则集中在Atom构造阶段完成。

#### 使用Coord坐标实现偏移的搬运

支持通过坐标参数实现有偏移的搬运操作：

```cpp
auto coord = MakeCoord(2, 3);
Copy(CopyAtom<CopyTraits<CopyL12L0A, CopyL12L0ATraitDefault>>{},
    l0aTensor, l1Tensor, coord);  // 从 l1Tensor 的 (2,3) 位置开始搬运
```

### 5.2 MmadAtom的构造与使用

MmadAtom的职责与CopyAtom相同，区别在于它封装的是矩阵计算类操作，而不是搬运类操作。

与CopyAtom不同，MmadAtom不仅有编译期Trait配置（MmadTrait），还通过MmadParams携带运行时维度参数（m/n/k、unitFlag、cmatrixInitVal 等），通过 with() 传入。

- MmadParams参数说明如下：

<table>
<thead>
<tr><th style="text-align: left;">参数名</th><th style="text-align: left;">类型</th><th style="text-align: left;">默认值</th><th style="text-align: left;">描述</th></tr>
</thead>
<tbody>
<tr><td style="text-align: left;">m</td><td style="text-align: left;">uint16_t</td><td style="text-align: left;">0</td><td style="text-align: left;">左矩阵A的高度，结果矩阵C的高度。</td></tr>
<tr><td style="text-align: left;">n</td><td style="text-align: left;">uint16_t</td><td style="text-align: left;">0</td><td style="text-align: left;">右矩阵B的宽度，结果矩阵C的宽度。</td></tr>
<tr><td style="text-align: left;">k</td><td style="text-align: left;">uint16_t</td><td style="text-align: left;">0</td><td style="text-align: left;">左矩阵A的宽度，右矩阵B的高度。</td></tr>
<tr><td style="text-align: left;">unitFlag</td><td style="text-align: left;">uint8_t</td><td style="text-align: left;">0</td><td style="text-align: left;">控制Mmad和后续矩阵数据搬出的细粒度并行。0表示不使能；2表示使能且执行后不复位单元标记位；3表示使能且执行后复位单元标记位。</td></tr>
<tr><td style="text-align: left;">cmatrixInitVal</td><td style="text-align: left;">bool</td><td style="text-align: left;">false</td><td style="text-align: left;">不传bias时，控制是否初始化结果矩阵C。true表示初始化后为零计算，false表示基于结果矩阵C原始值累加计算。</td></tr>
</tbody>
</table>

#### 定制化构造MmadAtom

定制MmadAtom时，需要区分两类参数：一类是编译期Trait，用于描述矩阵乘操作的固定特征；另一类是运行时MmadParams，用于描述本次调用的矩阵块大小和累加行为。通常建议先用MakeMmad构造一个可复用的MmadAtom，再在每次Mmad调用前用with()传入当前迭代需要的MmadParams。

定制步骤如下：

```cpp
// Step 1: 定义或选择矩阵乘的 Trait，字段以具体API支持的配置项为准
struct MyMmadTrait {
    // 在这里描述Mmad的编译期特征配置
};

// Step 2: 组合 MmadOperation 和 Trait，得到自定义 MmadAtom
auto myMmadAtom = MakeMmad(MmadOperation{}, MyMmadTrait{});

// Step 3: 根据当前矩阵块大小和是否首轮累加，构造运行时参数
MmadParams firstPara{baseM, baseN, baseK, 0, true};   // 首轮：初始化C矩阵
MmadParams accumPara{baseM, baseN, baseK, 0, false};  // 后续：在C上继续累加

// Step 4: 调用时通过with()绑定运行时参数
Mmad(myMmadAtom.with(firstPara), l0cTensor, l0aTensor, l0bTensor);
Mmad(myMmadAtom.with(accumPara), l0cTensor, l0aTensor, l0bTensor);
```

如果没有特殊的矩阵乘特征配置，可以使用默认Trait，只定制每次调用的MmadParams：

```cpp
auto defaultMmadAtom = MakeMmad(MmadOperation{}, MmadTraitDefault{});

for (uint32_t kIter = 0; kIter < kLoop; ++kIter) {
    MmadParams para{baseM, baseN, baseK, 0, kIter == 0};
    Mmad(defaultMmadAtom.with(para), l0cTensor, l0aTensor, l0bTensor);
}
```

这种写法的好处是：MmadAtom稳定描述"要做哪类矩阵乘"，MmadParams灵活描述"这一轮矩阵乘怎么做"。在K方向分块累加的场景中，只需要通过cmatrixInitVal区分首轮清零和后续累加，不需要反复重建Atom。

完整Mmad调用示例如下：

```cpp
// Step 1: 布置 L0A、L0B、L0C 张量
__ca__   half l0aBuf[baseM * baseK];
__cb__   half l0bBuf[baseK * baseN];
__cc__   float l0cBuf[baseM * baseN];

auto l0aTensor = MakeTensor(MakeMemPtr(l0aBuf), MakeFrameLayout<NZLayoutPtn>(baseM, baseK));
auto l0bTensor = MakeTensor(MakeMemPtr(l0bBuf), MakeFrameLayout<ZNLayoutPtn>(baseK, baseN));
auto l0cTensor = MakeTensor(MakeMemPtr(l0cBuf), MakeFrameLayout<NZLayoutPtn>(baseM, baseN));

// Step 2: 构造 MmadAtom
auto mmadAtom = MakeMmad(MmadOperation{}, MmadTraitDefault{});

// Step 3: 循环中按需传递 cmatrixInitVal
for (uint32_t kIter = 0; kIter < kLoop; kIter++) {
    // ... Copy L1→L0A, L1→L0B ...
    MmadParams para{baseM, baseN, baseK, 0, kIter == 0};
    Mmad(mmadAtom.with(para), l0cTensor, l0aTensor, l0bTensor);
}
```


在源文件中继续添加Atom对象的创建：

<div style="text-align: left;">
    <img src="./images/搬运计算流程.png" alt="搬运计算示意图" width="800">
    <p>图 6 搬运计算示意图 </p>
</div>

前面已经准备好了Layout、Tensor和Atom。现在进行矩阵块的搬运和计算，如图6所示，先将该核负责的完整A矩阵块（baseM × K）和B矩阵块（K × baseN）一次性从 GM 搬入 L1 缓存，随后进入K方向的分块循环。循环内每次迭代通过Slice沿K维度切出一个宽为baseK的子块：从A的L1 张量中取出坐标为 (0, kIter×baseK) ，大小为 (baseM, baseK) 的列切片送入L0A，从B中取出坐标为 (kIter×baseK, 0) ，大小为 (baseK, baseN) 的行切片送入L0B，二者在L0A Buffer和L0B Buffer上完成矩阵乘加并累加到 L0C Buffer上。其中首轮迭代通过MmadParams设置cmatrixInitVal=true清零矩阵C，后续轮次在已有结果上叠加（C += A_k × B_k）。借助 SetFlag/WaitFlag 的同步机制，形成搬运到计算的流水线。循环结束后，最终累加结果从L0C写回 GM，完成整个分块矩阵乘。


接下来，将完整的代码追加到源文件中：

In [ ]:
%%writefile -a Sources/matmul_tensor_api.asc

    constexpr uint32_t kLoop = K / baseK;

    // 将当前核完整的A/B矩阵块从GM搬入L1（一次性搬运）
    Copy(copyGM2L1Atom, l1ATensor, globalA);
    Copy(copyGM2L1Atom, l1BTensor, globalB);
    AscendC::SetFlag<AscendC::HardEvent::MTE2_MTE1>(EVENT_ID0);
    AscendC::WaitFlag<AscendC::HardEvent::MTE2_MTE1>(EVENT_ID0);
    AscendC::SetFlag<AscendC::HardEvent::M_MTE1>(EVENT_ID0);

    for (uint32_t kIter = 0; kIter < kLoop; ++kIter) {
        // 阶段1: L1 → L0A/L0B（MTE1流水）
        AscendC::WaitFlag<AscendC::HardEvent::M_MTE1>(EVENT_ID0);
        Copy(copyL12L0AAtom, l0ATensor,
            l1ATensor.Slice(MakeCoord(0, kIter * baseK), MakeShape(baseM, baseK)));
        Copy(copyL12L0BAtom, l0BTensor,
            l1BTensor.Slice(MakeCoord(kIter * baseK, 0), MakeShape(baseK, baseN)));

        AscendC::SetFlag<AscendC::HardEvent::MTE1_M>(EVENT_ID0);
        AscendC::WaitFlag<AscendC::HardEvent::MTE1_M>(EVENT_ID0);

        // 阶段2: 矩阵乘加（M流水）
        MmadParams para{baseM, baseN, baseK, 0, kIter == 0};
        Mmad(mmadAtom.with(para), l0CTensor, l0ATensor, l0BTensor);

        AscendC::SetFlag<AscendC::HardEvent::M_MTE1>(EVENT_ID0);
    }

    // 阶段3: L0C → GM（FIX流水），将计算结果写回全局内存
    AscendC::SetFlag<AscendC::HardEvent::M_FIX>(EVENT_ID0);
    AscendC::WaitFlag<AscendC::HardEvent::M_FIX>(EVENT_ID0);
    Copy(copyL0C2GMAtom, globalC, l0CTensor);

    // 等待尾流水结束
    AscendC::WaitFlag<AscendC::HardEvent::M_MTE1>(EVENT_ID0);
}

至此，核函数的设备侧实现已完成。下面开始编写Host侧应用程序来调用算子。

---

## 6. 核函数调用代码的开发

完成Kernel侧核函数开发后，即可编写Host侧的应用程序来调用算子并执行计算。矩阵乘算子的Host侧调用流程与[矢量Add算子](../../ascendc_operator_development/02_AscendC_basic/02.04_introduction_to_kernel_functions_based_on_add_operator.ipynb)的流程在结构上是相同的，但由于矩阵乘使用half类型数据，需要在数据准备和验证环节做相应适配。

### 6.1 kernel_matmul函数实现

kernel_matmul 函数负责在Host侧组织完整的核函数调用流程，主要包括以下七个步骤：
- 初始化AscendCL运行环境
- 申请与管理运行时资源（Device、Context、Stream）
- 在Host侧准备输入数据，并申请Host与Device侧内存，将数据从Host拷贝至Device
- 通过核函数调用符<<<...>>>在指定的Cube核上执行核函数
- 将计算结果从Device侧内存拷贝回Host侧
- 释放已申请的运行时资源
- 去初始化AscendCL环境

In [ ]:
%%writefile -a Sources/matmul_tensor_api.asc

std::vector<half> kernel_matmul(std::vector<half> &a, std::vector<half> &b)
{
    // ---------- 矩阵维度参数 ----------
    constexpr uint32_t M = 512;
    constexpr uint32_t K = 512;
    constexpr uint32_t N = 1024;
    constexpr uint32_t baseM = 128;
    constexpr uint32_t baseK = 64;
    constexpr uint32_t baseN = 256;
    constexpr uint32_t numBlocks = 16;

    size_t aByteSize = M * K * sizeof(half);
    size_t bByteSize = K * N * sizeof(half);
    size_t cByteSize = M * N * sizeof(half);

    // ---------- 初始化AscendCL运行环境 ----------
    int32_t deviceId = 0;
    aclrtStream stream = nullptr;

    aclInit(nullptr);
    aclrtSetDevice(deviceId);
    aclrtCreateStream(&stream);

    // ---------- 内存分配 ----------
    uint8_t *aHost = reinterpret_cast<uint8_t *>(a.data());
    uint8_t *bHost = reinterpret_cast<uint8_t *>(b.data());
    uint8_t *cHost = nullptr;
    uint8_t *aDevice = nullptr;
    uint8_t *bDevice = nullptr;
    uint8_t *cDevice = nullptr;

    aclrtMallocHost((void **)(&cHost), cByteSize);
    aclrtMalloc((void **)&aDevice, aByteSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc((void **)&bDevice, bByteSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc((void **)&cDevice, cByteSize, ACL_MEM_MALLOC_HUGE_FIRST);

    // ---------- 数据搬运（Host → Device） ----------
    aclrtMemcpy(aDevice, aByteSize, aHost, aByteSize, ACL_MEMCPY_HOST_TO_DEVICE);
    aclrtMemcpy(bDevice, bByteSize, bHost, bByteSize, ACL_MEMCPY_HOST_TO_DEVICE);

    // ---------- 核函数调用 ----------
    matmul_custom<M, K, N, baseM, baseM, baseK, baseN>
        <<<numBlocks, nullptr, stream>>>((half *)aDevice, (half *)bDevice, (half *)cDevice);
    aclrtSynchronizeStream(stream);

    // ---------- 结果回拷（Device → Host） ----------
    aclrtMemcpy(cHost, cByteSize, cDevice, cByteSize, ACL_MEMCPY_DEVICE_TO_HOST);
    std::vector<half> c((half *)cHost, (half *)(cHost + cByteSize));

    // ---------- 资源释放 ----------
    aclrtFree(aDevice);
    aclrtFree(bDevice);
    aclrtFree(cDevice);
    aclrtFreeHost(cHost);
    aclrtDestroyStream(stream);
    aclrtResetDevice(deviceId);
    aclFinalize();

    return c;
}

### 6.2 VerifyResult函数实现

VerifyResult函数中，将Ascend C算子计算的输出结果 (output) 与CPU侧通过相同逻辑计算出的预期值 (golden) 进行逐元素比对。矩阵乘使用half类型数据，比对时转换为float，并使用相对容差和绝对容差判断结果是否一致。

In [ ]:
%%writefile -a Sources/matmul_tensor_api.asc

uint32_t VerifyResult(std::vector<half> &output, std::vector<half> &golden)
{
    constexpr float RELATIVE_TOL = 1e-6f;
    constexpr float ABSOLUTE_TOL = 1e-9f;
    constexpr float ERROR_TOL = 1e-4f;

    size_t diffCount = 0;
    for (size_t i = 0; i < golden.size(); i++) {
        float outVal = static_cast<float>(output[i]);
        float goldVal = static_cast<float>(golden[i]);
        float absDiff = std::abs(outVal - goldVal);
        float relTol = RELATIVE_TOL * std::max(std::abs(outVal), std::abs(goldVal));
        if (absDiff > std::max(ABSOLUTE_TOL, relTol)) {
            diffCount++;
            if (diffCount <= 100) {
                std::cout << "data index: " << i << ", expected: " << goldVal
                          << ", actual: " << outVal << ", abs_diff: " << absDiff << std::endl;
            }
        }
    }

    float errorRatio = static_cast<float>(diffCount) / golden.size();
    std::cout << "error ratio: " << errorRatio << ", tolerance: " << ERROR_TOL << std::endl;

    if (errorRatio <= ERROR_TOL) {
        std::cout << "[Success] Case accuracy is verification passed." << std::endl;
        return 0;
    } else {
        std::cout << "[Failed] Case accuracy is verification failed!" << std::endl;
        return 1;
    }
}

### 6.3 Host侧验证主程序main函数

main函数负责生成随机输入数据、在CPU侧计算golden结果、调用算子并完成验证。

In [ ]:
%%writefile -a Sources/matmul_tensor_api.asc

int32_t main(int32_t argc, char *argv[])
{
    constexpr uint32_t M = 512;
    constexpr uint32_t K = 512;
    constexpr uint32_t N = 1024;
    uint32_t aSize = M * K;
    uint32_t bSize = K * N;
    uint32_t cSize = M * N;

    // ---------- 生成随机输入数据 ----------
    std::mt19937 rng(42);
    std::uniform_int_distribution<int32_t> dist(-10, 10);
    std::vector<half> a(aSize);
    std::vector<half> b(bSize);
    for (uint32_t i = 0; i < aSize; i++) {
        a[i] = static_cast<half>(static_cast<float>(dist(rng)));
    }
    for (uint32_t i = 0; i < bSize; i++) {
        b[i] = static_cast<half>(static_cast<float>(dist(rng)));
    }

    // ---------- CPU计算golden结果 ----------
    std::vector<float> aFloat(aSize);
    std::vector<float> bFloat(bSize);
    for (uint32_t i = 0; i < aSize; i++) {
        aFloat[i] = static_cast<float>(a[i]);
    }
    for (uint32_t i = 0; i < bSize; i++) {
        bFloat[i] = static_cast<float>(b[i]);
    }

    std::vector<half> golden(cSize);
    for (uint32_t m = 0; m < M; m++) {
        for (uint32_t n = 0; n < N; n++) {
            float sum = 0.0f;
            for (uint32_t k = 0; k < K; k++) {
                sum += aFloat[m * K + k] * bFloat[k * N + n];
            }
            golden[m * N + n] = static_cast<half>(sum);
        }
    }

    // ---------- 调用核函数 ----------
    std::vector<half> output = kernel_matmul(a, b);

    // ---------- 结果验证 ----------
    return VerifyResult(output, golden);
}

---

## 7. CMake编译配置

一个完整的算子应用程序需要通过编译生成可执行文件。对于Tensor API项目，我们需要配置CMake以支持Ascend C编译工具链，并指定目标架构为dav-3510（Ascend 950PR/Ascend 950DT）。

In [ ]:
%%writefile Sources/CMakeLists.txt

cmake_minimum_required(VERSION 3.16)

set(CMAKE_ASC_RUN_MODE "npu" CACHE STRING "Run mode: npu, sim")
set(CMAKE_ASC_ARCHITECTURES "dav-3510" CACHE STRING "NPU architecture: dav-3510")

find_package(ASC REQUIRED)
project(kernel_samples LANGUAGES ASC CXX)

add_executable(demo
    matmul_tensor_api.asc
)

set(ASCEND_HOME $ENV{ASCEND_HOME_PATH})
set(ASC_INCLUDE_DIR "${ASCEND_HOME}/asc/")
include_directories(
    "${ASC_INCLUDE_DIR}"
)

target_compile_options(demo PRIVATE
    $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=3510>
)

---

## 8. 编译和运行

执行以下命令编译可执行文件：

In [ ]:
!cd Sources && mkdir -p build                                              # 创建并进入build目录
cmake -DCMAKE_ASC_ARCHITECTURES=dav-3510 ..;make -j;                      # 编译工程（默认npu模式）                                                                # 执行编译生成的可执行程序，执行样例

编译成功后，执行以下代码运行算子：

In [ ]:
!cd Sources/build/ && ./demo

---

## 9. 小结

本节系统介绍了Tensor API的分层架构与核心编程模型。Tensor API将张量的构造拆解为两个环节：通过MakeMemPtr统一创建带内存位置标签的硬件指针，通过MakeFrameLayout描述分形的形状、步幅，二者由MakeTensor组合为Tensor对象，可通过Slice实现张量的偏移与子张量的构造。

在计算与搬运层面，Tensor API通过Atom层将操作类别与特征参数（Trait）封装为的原子对象：CopyAtom覆盖了从GM到L0C的全部搬运路径；MmadAtom封装矩阵乘加操作；Arch层则根据布局模式、数据类型和架构版本，通过routing机制静态分发到对应的硬件指令路径。

通过完整的矩阵乘样例开发实践，我们掌握了从Layout定义、Tensor构建、Atom创建到组装的完整Tensor API编程范式，以及Host侧的调用流程。

整体来看，Tensor API通过模板编程，将Shape/Stride的计算、Trait的模板匹配、架构路径的静态路由在编译阶段完成，提供了高可读性坐标式编程体验。

---


## 10. 附录

本节主要聚焦基于tensor的C++编程模型、分层设计与矩阵乘样例。若需要进一步查阅详细接口说明，可参考以下官方文档链接：

- 基础数据结构：<https://gitcode.com/cann/asc-devkit/tree/master/docs/api/SIMD-API/%E5%9F%BA%E7%A1%80API/%E6%95%B0%E6%8D%AE%E7%BB%93%E6%9E%84/TensorAPI-BasicStructure>
- 搬运和计算接口：<https://gitcode.com/cann/asc-devkit/tree/master/docs/api/SIMD-API/%E5%9F%BA%E7%A1%80API/%E7%9F%A9%E9%98%B5%E8%AE%A1%E7%AE%97%EF%BC%88TensorAPI%EF%BC%89>

建议阅读顺序如下：

1. 先阅读“基础数据结构”，理解 MakeFrameLayout、MakeMemPtr、MakeTensor、Slice 等对象和接口之间的关系。
2. 再阅读“搬运和计算接口”，结合本文中的 CopyAtom、MmadAtom 和 routing 机制理解 Tensor API 如何完成数据搬运与矩阵计算。

---


## 课后实践

### 1. 选择题

**Q1.** 以下哪项不是Tensor API中CopyAtom 支持的Trait类型？
- A. DataCopyTrait——不带配置参数的纯标记 Trait
- B. LoadDataTrait——带 transposed 转置控制的 Trait
- C. FixpipeTrait——带 ReLU/通道拆分后处理的 Trait
- D. MmadTrait——带矩阵乘编译期配置的 Trait

**Q2.** MmadParams::cmatrixInitVal的正确使用方式是？
- A. 每次 Mmad 调用都设为true
- B. K方向第一次迭代设为true，后续迭代设为false（累加）
- C. K方向第一次迭代设为false，后续迭代设为true
- D. 永远设为false

**Q3.** 以下关于MakeFrameLayout的描述，正确的是？
- A. MakeFrameLayout<NZLayoutPtn>(M, K)创建的布局与 MakeFrameLayout<NDExtLayoutPtn>(M, K) 完全相同
- B. MakeFrameLayout<NZLayoutPtn>() 创建的布局内部是四维分形结构
- C. MakeFrameLayout必须在每次运行时动态计算 Shape 和 Stride，产生运行时开销
- D. MakeFrameLayout的 TraitType 参数决定了数据的元素类型

### 2. 判断题

**Q4.** Atom 的 with() 方法会修改调用它的原始 Atom 对象。
- A. 正确
- B. 错误

**Q5.** 在 Tensor API 的三层架构中，Atom层进行routing分发机制。
- A. 正确
- B. 错误

---
**执行以下代码获取答案。**

In [ ]:
!cat ./answer/03.03.05_answer.txt